# Phase 6: Interpretation Artifacts

1. **Variance decomposition** — parent vs scenario random-effect SDs for both stages
2. **Stable feature shortlist** — features where hierarchical CI excludes 0 + enet selected + sign agrees
3. **Qualitative triangulation** — TF-IDF distinguishing terms for top/bottom sentiment terciles
4. **Wave 2 preregistration draft** — surviving hypotheses, suggested N

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'arviz', 'scikit-learn'], check=False)

import os, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az

SEED = 20260506
np.random.seed(SEED)

_nb_dir = os.path.abspath('.')
TRACES_DIR = os.path.join(_nb_dir, 'traces')

# df_sel_aug: 200 rows (one per unique selection) — for model output / validation
with open(os.path.join(TRACES_DIR, 'df_sel_aug.pkl'), 'rb') as f:
    df_sel_aug = pickle.load(f)
# df_sel: 267 rows (one per selection × concern_item) — for rationale_text analysis
with open(os.path.join(TRACES_DIR, 'df_sel.pkl'), 'rb') as f:
    df_sel_full = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'validation_results.pkl'), 'rb') as f:
    val = pickle.load(f)

trace_a, trace_b = None, None
try:
    trace_a = az.from_netcdf(os.path.join(TRACES_DIR, 'stage_a_hier.nc'))
    print('Stage A trace loaded.')
except Exception as e:
    print(f'Stage A trace not available: {e}')
try:
    trace_b = az.from_netcdf(os.path.join(TRACES_DIR, 'stage_b.nc'))
    print('Stage B trace loaded.')
except Exception as e:
    print(f'Stage B trace not available: {e}')

try:
    with open(os.path.join(TRACES_DIR, 'stage_a_comparison.pkl'), 'rb') as f:
        comparison_a = pickle.load(f)
except:
    comparison_a = None

try:
    with open(os.path.join(TRACES_DIR, 'hier_coefs_b.pkl'), 'rb') as f:
        hier_coefs_b = pickle.load(f)
except:
    hier_coefs_b = None

print(f'df_sel_aug (modeling): {df_sel_aug.shape}')
print(f'df_sel_full (rationale text): {df_sel_full.shape}')

## 6.1  Variance decomposition (primary deliverable)

In [ ]:
def extract_variance_decomp(trace, label):
    """Extract parent SD, scenario SD from trace posterior."""
    sd_vars = [v for v in trace.posterior.data_vars if 'sigma' in v.lower() or '1|' in v]
    if not sd_vars:
        print(f'{label}: No sigma variables found in trace.')
        return None

    rows = []
    for v in sd_vars:
        vals = trace.posterior[v].values.flatten()
        rows.append({
            'variable': v,
            'mean_sd': vals.mean(),
            'sd_sd': vals.std(),
            'hdi_lo': np.percentile(vals, 3),
            'hdi_hi': np.percentile(vals, 97),
        })
    df = pd.DataFrame(rows)
    df.insert(0, 'stage', label)
    return df


decomp_rows = []
if trace_a is not None:
    d = extract_variance_decomp(trace_a, 'Stage A (selection)')
    if d is not None:
        decomp_rows.append(d)
if trace_b is not None:
    d = extract_variance_decomp(trace_b, 'Stage B (sentiment)')
    if d is not None:
        decomp_rows.append(d)

if decomp_rows:
    decomp_df = pd.concat(decomp_rows, ignore_index=True)
    print('=== VARIANCE DECOMPOSITION (Primary Deliverable) ===')
    display(decomp_df.round(3))

    # Bar chart
    fig, axes = plt.subplots(1, max(1, len(decomp_rows)), figsize=(4 * max(1, len(decomp_rows)), 4))
    if len(decomp_rows) == 1:
        axes = [axes]
    for ax, d in zip(axes, decomp_rows):
        ax.barh(d['variable'], d['mean_sd'], xerr=d['sd_sd'],
                color='steelblue', alpha=0.7)
        ax.set_title(d['stage'].iloc[0])
        ax.set_xlabel('Random-effect SD (posterior mean ± SD)')
    plt.tight_layout()
    plt.show()
else:
    print('No traces available for variance decomposition. Run notebooks 2 and 3 first.')

## 6.2  Stable feature shortlist (Wave 2 hypotheses)

In [ ]:
# Stage A: features where hier CI excludes 0 AND enet selected AND sign agrees
if comparison_a is not None:
    wave2_a = comparison_a[comparison_a['agreement_flag']].copy()
    wave2_a = wave2_a.sort_values('hier_mean', key=abs, ascending=False)
    print(f'=== Stage A: Wave 2 candidate features ({len(wave2_a)}) ===')
    display(wave2_a[['hier_mean', 'hier_ci_lo', 'hier_ci_hi', 'enet_coef']].round(3))
else:
    wave2_a = pd.DataFrame()
    print('Stage A comparison not available.')

# Stage B: features where hier CI excludes 0 (no enet comparison available)
if hier_coefs_b is not None:
    ci_lo_col = [c for c in hier_coefs_b.columns if 'hdi_3' in c or 'ci_lo' in c]
    ci_hi_col = [c for c in hier_coefs_b.columns if 'hdi_97' in c or 'ci_hi' in c]
    if ci_lo_col and ci_hi_col:
        wave2_b = hier_coefs_b[
            (hier_coefs_b[ci_lo_col[0]] > 0) | (hier_coefs_b[ci_hi_col[0]] < 0)
        ].copy()
        wave2_b = wave2_b.sort_values('mean', key=abs, ascending=False)
        print(f'\n=== Stage B: Wave 2 candidate features (CI excludes 0, {len(wave2_b)}) ===')
        display(wave2_b[['mean', ci_lo_col[0], ci_hi_col[0]]].head(20).round(3))
    else:
        print('Stage B: CI columns not found in summary.')
else:
    print('Stage B coefficients not available.')

## 6.3  Qualitative triangulation (TF-IDF on rationale text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use df_sel_full (267 rows) — all rationale blocks, including multiple per highlight.
# This maximises text coverage for the descriptive TF-IDF analysis.
df_text = df_sel_full[['highlight_sentiment', 'rationale_text']].dropna(subset=['rationale_text'])
df_text = df_text[df_text['rationale_text'].str.strip() != '']

top_tercile    = df_text[df_text['highlight_sentiment'] >= 6]['rationale_text'].tolist()
bottom_tercile = df_text[df_text['highlight_sentiment'] <= 3]['rationale_text'].tolist()

print(f'Top tercile (sentiment ≥ 6):    {len(top_tercile)} rationale blocks')
print(f'Bottom tercile (sentiment ≤ 3): {len(bottom_tercile)} rationale blocks')

if len(top_tercile) > 3 and len(bottom_tercile) > 3:
    all_texts = top_tercile + bottom_tercile
    labels    = [1] * len(top_tercile) + [0] * len(bottom_tercile)

    vec = TfidfVectorizer(max_features=500, ngram_range=(1, 2),
                          stop_words='english', min_df=2)
    X_tfidf = vec.fit_transform(all_texts).toarray()
    terms = np.array(vec.get_feature_names_out())

    pos_mean = X_tfidf[np.array(labels) == 1].mean(axis=0)
    neg_mean = X_tfidf[np.array(labels) == 0].mean(axis=0)
    log_odds  = np.log((pos_mean + 1e-9) / (neg_mean + 1e-9))

    top20_pos = (pd.DataFrame({'term': terms, 'log_odds': log_odds})
                 .sort_values('log_odds', ascending=False).head(20).reset_index(drop=True))
    top20_neg = (pd.DataFrame({'term': terms, 'log_odds': log_odds})
                 .sort_values('log_odds').head(20).reset_index(drop=True))

    print('\nTop 20 terms distinctive of HIGH sentiment (≥ 6):')
    display(top20_pos)
    print('\nTop 20 terms distinctive of LOW sentiment (≤ 3):')
    display(top20_neg)
else:
    print('Insufficient data for TF-IDF analysis.')

## 6.4  Key findings summary and Wave 2 preregistration draft

In [ ]:
# ── Validation summary ─────────────────────────────────────────────────────
print('=== VALIDATION SUMMARY ===')
print('\nStage A (selection) vs baselines:')
display(val['baseline_a'].round(4))
print('\nStage B (sentiment) vs baselines:')
display(val['baseline_b'].round(4))

## Wave 2 Preregistration Draft

**Study**: KidsGPT Parent Highlight Behavior — Wave 2 Confirmatory Study

**Background**: Pilot (N=20 parents, ~45 scenarios, 267 highlights) identified candidate
predictors of (a) which AI-response sentences parents highlight and (b) their sentiment
rating. See stable feature shortlist above.

**Hypotheses** (to be filled after reviewing stable feature shortlist above):
- H1 (Stage A): [Feature X] positively predicts highlight probability (β > 0, 95% CI excludes 0)
- H2 (Stage A): [Feature Y] ...
- H3 (Stage B): [Feature Z] positively predicts sentiment rating

**Primary outcomes**:
- Stage A: PR-AUC in LOPO-CV > baseline per-span empirical rate
- Stage B: Spearman ρ in LOPO-CV > per-parent mean baseline

**Required effect size** (to be derived from pilot posterior means above):
- Minimum detectable effect: OR ≥ 1.5 for Stage A, β ≥ 0.5 SD for Stage B

**Suggested N**: ~80 parents × 45 scenarios to achieve 80% power at α=0.05
(1/4× parent random-effect variance, 4× sample size).

**Caveats from pilot**:
- N=20 parents, convenience sample from Prolific
- Single highlight per span (no redundancy)
- Scenarios limited to one AI response per prompt (no multi-turn)

In [ ]:
print('Interpretation notebook complete.')
print('Key artifacts displayed above:')
print('  6.1 Variance decomposition (parent SD vs scenario SD)')
print('  6.2 Stable feature shortlist (Wave 2 hypotheses)')
print('  6.3 TF-IDF distinguishing terms by sentiment tercile')
print('  6.4 Validation summary + Wave 2 preregistration draft')